In [0]:
%run ./00_config_setup

#### 1. S3 bucket path

## Utility_functions

In [0]:
SCHEMA = BRONZE_SCHEMA
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")


def sanitise_columns(df: DataFrame) -> DataFrame:
    clean = [
        re.sub(r"[^a-zA-Z0-9]", "_", c).lower()
        for c in df.columns
    ]
    return df.toDF(*clean)


def add_bronze_metadata(df: DataFrame, domain: str, source_path: str) -> DataFrame:
    return (
        df
        .withColumn("ingestion_timestamp", current_timestamp())
        .withColumn("ingestion_date",      to_date(current_timestamp()))
        .withColumn("batch_id",            lit(BATCH_ID))
        .withColumn("source_file_name",    lit(source_path))
        .withColumn("domain",              lit(domain))
    )


def write_bronze_external(
    df: DataFrame,
    table_name: str,
    s3_path: str,
    partition_col: str = "ingestion_date"
) -> int:

    (
        df.write
        .format("delta")
        .mode("append") 
        .option("mergeSchema", "true")
        .option("maxRecordsPerFile", 500000)   
        .partitionBy(partition_col)
        .save(s3_path)
    )

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {table_name}
        USING DELTA
        LOCATION '{s3_path}'
    """)

    row_count = spark.table(table_name).count()
    return row_count


def show_delta_history(table_name: str, limit: int = 3):
    print(f"\n  Delta History — {table_name} (last {limit} ops):")
    
    (
        spark.sql(f"DESCRIBE HISTORY {table_name}")
        .orderBy("version", ascending=False)
        .limit(limit)
        .select("version", "timestamp", "operation", "operationParameters")
        .show(truncate=False)
    )


def show_table_details(table_name: str):
    print(f"\n  Delta Detail — {table_name}:")
    
    (
        spark.sql(f"DESCRIBE DETAIL {table_name}")
        .select("name", "location", "numFiles", "sizeInBytes", "partitionColumns")
        .show(truncate=False)
    )

def get_s3_path_size(path):
    files = dbutils.fs.ls(path)
    total_size = sum(f.size for f in files)
    return total_size


def calculate_partitions(size_bytes, target_mb=128):
    size_mb = size_bytes / (1024 * 1024)
    partitions = max(1, int(size_mb / target_mb))
    return partitions


def apply_dynamic_partitioning(df, partitions):
    if partitions <= 5:
        print("🔹 Small dataset → using coalesce")
        return df.coalesce(partitions)
    else:
        return df.repartition(partitions)


## DOMAIN 1 — ELECTRONICS


#### 1. read

In [0]:
df_elec_raw = (
    spark.read
    .format("csv")
    .option("header",      "true")
    .option("delimiter",   ",")
    .option("inferSchema", "false")   
    .option("encoding",    "UTF-8")
    .load(S3_RAW_ELECTRONICS)
)


print(f"  Raw rows loaded : {df_elec_raw.count():,}")
print(f"  Raw columns     : {len(df_elec_raw.columns)}")
df_elec_raw.printSchema()


  Raw rows loaded : 186,850
  Raw columns     : 6
root
 |-- Order ID: string (nullable = true)
 |-- Product: string (nullable = true)
 |-- Quantity Ordered: string (nullable = true)
 |-- Price Each: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Purchase Address: string (nullable = true)



#### 2. metadata

In [0]:
df_elec = df_elec_raw.withColumn(
    "_raw_file_path", col("_metadata.file_path")
)

df_elec = add_bronze_metadata(
    df_elec,
    domain      = "electronics",
    source_path = S3_RAW_ELECTRONICS
)

df_elec = df_elec.withColumn(
    "source_month",
    regexp_extract(col("_raw_file_path"), r"Sales_([A-Za-z]+_\d{4})\.csv", 1)
)

df_elec = df_elec.drop("_raw_file_path")
df_elec = sanitise_columns(df_elec)

print(f"\n  Schema after metadata enrichment:")
df_elec.printSchema()


  Schema after metadata enrichment:
root
 |-- order_id: string (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity_ordered: string (nullable = true)
 |-- price_each: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- purchase_address: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = false)
 |-- ingestion_date: date (nullable = false)
 |-- batch_id: string (nullable = false)
 |-- source_file_name: string (nullable = false)
 |-- domain: string (nullable = false)
 |-- source_month: string (nullable = false)



#### 3. write + dynamic repartition

In [0]:

# dynamic repartition
size_bytes = get_s3_path_size(S3_RAW_ELECTRONICS)
partitions = calculate_partitions(size_bytes)
df_elec = apply_dynamic_partitioning(df_elec, partitions)


#write to the s3
elec_rows = write_bronze_external(
    df          = df_elec,
    table_name  = TBL_BRZ_ELECTRONICS,
    s3_path     = S3_BRONZE_ELECTRONICS,
    partition_col = "ingestion_date"
)


show_delta_history(TBL_BRZ_ELECTRONICS)
show_table_details(TBL_BRZ_ELECTRONICS)

print(f"\n  Sample — Electronics Bronze:")
display(spark.table(TBL_BRZ_ELECTRONICS).limit(5))

🔹 Small dataset → using coalesce

  Delta History — retailflow360.bronze.electronics (last 3 ops):
+-------+-------------------+---------+-------------------------------------------------------------------------+
|version|timestamp          |operation|operationParameters                                                      |
+-------+-------------------+---------+-------------------------------------------------------------------------+
|1      |2026-04-22 09:31:16|WRITE    |{mode -> Append, statsOnLoad -> false, partitionBy -> ["ingestion_date"]}|
|0      |2026-04-22 08:38:24|WRITE    |{mode -> Append, statsOnLoad -> false, partitionBy -> ["ingestion_date"]}|
+-------+-------------------+---------+-------------------------------------------------------------------------+


  Delta Detail — retailflow360.bronze.electronics:
+--------------------------------+-----------------------------------------------+--------+-----------+----------------+
|name                            |location 

order_id,product,quantity_ordered,price_each,order_date,purchase_address,ingestion_timestamp,ingestion_date,batch_id,source_file_name,domain,source_month
295665,Macbook Pro Laptop,1,1700,12/30/19 00:01,"136 Church St, New York City, NY 10001",2026-04-22T09:31:11.315Z,2026-04-22,5658f921-d318-40ee-8c4b-1a5ba5c57c88,s3://retailflow360-capstone3/raw/electronics/,electronics,December_2019
295666,LG Washing Machine,1,600.0,12/29/19 07:03,"562 2nd St, New York City, NY 10001",2026-04-22T09:31:11.315Z,2026-04-22,5658f921-d318-40ee-8c4b-1a5ba5c57c88,s3://retailflow360-capstone3/raw/electronics/,electronics,December_2019
295667,USB-C Charging Cable,1,11.95,12/12/19 18:21,"277 Main St, New York City, NY 10001",2026-04-22T09:31:11.315Z,2026-04-22,5658f921-d318-40ee-8c4b-1a5ba5c57c88,s3://retailflow360-capstone3/raw/electronics/,electronics,December_2019
295668,27in FHD Monitor,1,149.99,12/22/19 15:13,"410 6th St, San Francisco, CA 94016",2026-04-22T09:31:11.315Z,2026-04-22,5658f921-d318-40ee-8c4b-1a5ba5c57c88,s3://retailflow360-capstone3/raw/electronics/,electronics,December_2019
295669,USB-C Charging Cable,1,11.95,12/18/19 12:38,"43 Hill St, Atlanta, GA 30301",2026-04-22T09:31:11.315Z,2026-04-22,5658f921-d318-40ee-8c4b-1a5ba5c57c88,s3://retailflow360-capstone3/raw/electronics/,electronics,December_2019


## DOMAIN 2 — LIQUOR SALES

#### 1. read

In [0]:
df_liquor_raw = (
    spark.read
    .format("csv")
    .option("header",      "true")
    .option("inferSchema", "false")    
    .option("encoding",    "UTF-8")
    .option("multiLine",   "false")   
    .option("quote",       '"')
    .option("escape",      '"')
    .load(S3_RAW_LIQUOR)
)

print(f"  Raw rows loaded : {df_liquor_raw.count():,}")
print(f"  Raw columns     : {len(df_liquor_raw.columns)}")
df_liquor_raw.printSchema()


  Raw rows loaded : 19,666,763
  Raw columns     : 24
root
 |-- Invoice/Item Number: string (nullable = true)
 |-- Date: string (nullable = true)
 |-- Store Number: string (nullable = true)
 |-- Store Name: string (nullable = true)
 |-- Address: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Zip Code: string (nullable = true)
 |-- Store Location: string (nullable = true)
 |-- County Number: string (nullable = true)
 |-- County: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Category Name: string (nullable = true)
 |-- Vendor Number: string (nullable = true)
 |-- Vendor Name: string (nullable = true)
 |-- Item Number: string (nullable = true)
 |-- Item Description: string (nullable = true)
 |-- Pack: string (nullable = true)
 |-- Bottle Volume (ml): string (nullable = true)
 |-- State Bottle Cost: string (nullable = true)
 |-- State Bottle Retail: string (nullable = true)
 |-- Bottles Sold: string (nullable = true)
 |-- Sale (Dollars): string (n

#### 2. metadata

In [0]:
df_liquor_raw = df_liquor_raw.withColumn("_raw_file_path", col("_metadata.file_path"))


df_liquor = add_bronze_metadata(
    df_liquor_raw,
    domain      = "liquor",
    source_path = S3_RAW_LIQUOR
)
df_liquor = sanitise_columns(df_liquor)

print(f"\n  Schema after metadata enrichment:")
df_liquor.printSchema()



  Schema after metadata enrichment:
root
 |-- invoice_item_number: string (nullable = true)
 |-- date: string (nullable = true)
 |-- store_number: string (nullable = true)
 |-- store_name: string (nullable = true)
 |-- address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- zip_code: string (nullable = true)
 |-- store_location: string (nullable = true)
 |-- county_number: string (nullable = true)
 |-- county: string (nullable = true)
 |-- category: string (nullable = true)
 |-- category_name: string (nullable = true)
 |-- vendor_number: string (nullable = true)
 |-- vendor_name: string (nullable = true)
 |-- item_number: string (nullable = true)
 |-- item_description: string (nullable = true)
 |-- pack: string (nullable = true)
 |-- bottle_volume__ml_: string (nullable = true)
 |-- state_bottle_cost: string (nullable = true)
 |-- state_bottle_retail: string (nullable = true)
 |-- bottles_sold: string (nullable = true)
 |-- sale__dollars_: string (nullable = true)
 

#### 3. write + dynamic repartition

In [0]:
# dynamic repartition
size_bytes = get_s3_path_size(S3_RAW_LIQUOR)
partitions = calculate_partitions(size_bytes)
df_liquor = apply_dynamic_partitioning(df_liquor, partitions)


# Write external Delta table to S3
liquor_rows = write_bronze_external(
    df            = df_liquor,
    table_name    = TBL_BRZ_LIQUOR,
    s3_path       = S3_BRONZE_LIQUOR,
    partition_col = "ingestion_date"
)


# Delta time travel audit
show_delta_history(TBL_BRZ_LIQUOR)
show_table_details(TBL_BRZ_LIQUOR)

print(f"  Sample — Liquor Bronze:")
display(spark.table(TBL_BRZ_LIQUOR).limit(5))


  Delta History — retailflow360.bronze.liquor_sales (last 3 ops):
+-------+-------------------+---------+-------------------------------------------------------------------------+
|version|timestamp          |operation|operationParameters                                                      |
+-------+-------------------+---------+-------------------------------------------------------------------------+
|1      |2026-04-22 09:32:40|WRITE    |{mode -> Append, statsOnLoad -> false, partitionBy -> ["ingestion_date"]}|
|0      |2026-04-22 08:40:09|WRITE    |{mode -> Append, statsOnLoad -> false, partitionBy -> ["ingestion_date"]}|
+-------+-------------------+---------+-------------------------------------------------------------------------+


  Delta Detail — retailflow360.bronze.liquor_sales:
+---------------------------------+------------------------------------------------+--------+-----------+----------------+
|name                             |location                             

invoice_item_number,date,store_number,store_name,address,city,zip_code,store_location,county_number,county,category,category_name,vendor_number,vendor_name,item_number,item_description,pack,bottle_volume__ml_,state_bottle_cost,state_bottle_retail,bottles_sold,sale__dollars_,volume_sold__liters_,volume_sold__gallons_,_raw_file_path,ingestion_timestamp,ingestion_date,batch_id,source_file_name,domain
381800108,02/29/2012,3818,Round Window Liquor,703 W PLEASANT ST,KNOXVILLE,50138,POINT (-93.10564100000002 41.320746),63,Marion,1012100,CANADIAN WHISKIES,115,"Constellation Wine Company, Inc.",11773,Black Velvet,48,200,1.56,2.34,8,18.72,1.60,0.42,s3://retailflow360-capstone3/raw/liquor/Liquor_Sales.csv,2026-04-22T08:38:50.801Z,2026-04-22,ccc4ff1d-cec7-4dab-a79f-fbbaf430d6b7,s3://retailflow360-capstone3/raw/liquor/Liquor_Sales.csv,liquor
INV-00001300074,08/29/2016,2651,Hy-Vee / Waverly,1311 4 Street SW,Waverly,50677,POINT (-92.475403 42.712263),9,BREMER,1081200,Cream Liqueurs,330,Gemini Spirits,80458,Ryan's Cream Liqueur,6,1750,11.54,16.31,1,97.86,1.75,0.46,s3://retailflow360-capstone3/raw/liquor/Liquor_Sales.csv,2026-04-22T08:38:50.801Z,2026-04-22,ccc4ff1d-cec7-4dab-a79f-fbbaf430d6b7,s3://retailflow360-capstone3/raw/liquor/Liquor_Sales.csv,liquor
INV-00003500038,08/29/2016,5128,Price Chopper / Beaver,1819 Beaver Ave,Des Moines,50310,POINT (-93.672695 41.610525),77,POLK,1031000,American Vodka,300,McCormick Distilling Co.,36886,Mccormick Vodka,12,750,3.31,4.97,2,59.64,1.50,0.39,s3://retailflow360-capstone3/raw/liquor/Liquor_Sales.csv,2026-04-22T08:38:50.801Z,2026-04-22,ccc4ff1d-cec7-4dab-a79f-fbbaf430d6b7,s3://retailflow360-capstone3/raw/liquor/Liquor_Sales.csv,liquor
INV-00004900133,08/29/2016,2569,Hy-Vee Food Store #2 / Cedar Rapids,279 Collins Road NE,Cedar Rapids,52402,POINT (-91.630677 42.027805),57,LINN,1070000,Cocktails / RTD,055,SAZERAC NORTH AMERICA,57125,Chi-Chi's Gold Margarita,6,1750,6.50,9.75,1,58.50,1.75,0.46,s3://retailflow360-capstone3/raw/liquor/Liquor_Sales.csv,2026-04-22T08:38:50.801Z,2026-04-22,ccc4ff1d-cec7-4dab-a79f-fbbaf430d6b7,s3://retailflow360-capstone3/raw/liquor/Liquor_Sales.csv,liquor
INV-00005400009,08/29/2016,4051,Fareway Stores #639 / Maquoketa,110 Westgate Dr,Maquoketa,52060,POINT (-90.680623 42.067391),49,JACKSON,1041100,American Dry Gins,434,LUXCO INC,31656,Paramount Gin,12,750,3.40,5.10,1,61.20,0.75,0.19,s3://retailflow360-capstone3/raw/liquor/Liquor_Sales.csv,2026-04-22T08:38:50.801Z,2026-04-22,ccc4ff1d-cec7-4dab-a79f-fbbaf430d6b7,s3://retailflow360-capstone3/raw/liquor/Liquor_Sales.csv,liquor


## DOMAIN 3 — BOOKS DATA (catalog metadata)

#### 1. read

In [0]:
df_books_data_raw = (
    spark.read
    .format("csv")
    .option("header",      "true")
    .option("inferSchema", "false")
    .option("encoding",    "UTF-8")
    .option("multiLine",   "true")     # descriptions contain newlines
    .option("quote",       '"')
    .option("escape",      '"')
    .load(S3_RAW_BOOKS_DATA)
)

print(f"  Raw rows loaded : {df_books_data_raw.count():,}")
print(f"  Raw columns     : {len(df_books_data_raw.columns)}")
df_books_data_raw.printSchema()


  Raw rows loaded : 212,404
  Raw columns     : 10
root
 |-- Title: string (nullable = true)
 |-- description: string (nullable = true)
 |-- authors: string (nullable = true)
 |-- image: string (nullable = true)
 |-- previewLink: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- publishedDate: string (nullable = true)
 |-- infoLink: string (nullable = true)
 |-- categories: string (nullable = true)
 |-- ratingsCount: string (nullable = true)



#### 2. metadata

In [0]:
df_books_data_raw = df_books_data_raw.withColumn("_raw_file_path", col("_metadata.file_path"))


df_books_data = add_bronze_metadata(
    df_books_data_raw,
    domain      = "books_data",
    source_path = S3_RAW_BOOKS_DATA
)
df_books_data = sanitise_columns(df_books_data)

print(f"\n  Schema after metadata enrichment:")
df_books_data.printSchema()


  Schema after metadata enrichment:
root
 |-- title: string (nullable = true)
 |-- description: string (nullable = true)
 |-- authors: string (nullable = true)
 |-- image: string (nullable = true)
 |-- previewlink: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- publisheddate: string (nullable = true)
 |-- infolink: string (nullable = true)
 |-- categories: string (nullable = true)
 |-- ratingscount: string (nullable = true)
 |-- _raw_file_path: string (nullable = false)
 |-- ingestion_timestamp: timestamp (nullable = false)
 |-- ingestion_date: date (nullable = false)
 |-- batch_id: string (nullable = false)
 |-- source_file_name: string (nullable = false)
 |-- domain: string (nullable = false)



#### 3. write + dynamic repartition

In [0]:
# dynamic repartition
size_bytes = get_s3_path_size(S3_RAW_BOOKS_DATA)
partitions = calculate_partitions(size_bytes)
df_books_data = apply_dynamic_partitioning(df_books_data, partitions)

# Write external Delta table to S3 # 
books_data_rows = write_bronze_external(
    df            = df_books_data,
    table_name    = TBL_BRZ_BOOKS_DATA,
    s3_path       = S3_BRONZE_BOOKS_DATA,
    partition_col = "ingestion_date"
)

# Delta time travel audit
show_delta_history(TBL_BRZ_BOOKS_DATA)
show_table_details(TBL_BRZ_BOOKS_DATA)

print(f"  Sample — Books Data Bronze:")
display(spark.table(TBL_BRZ_BOOKS_DATA).limit(5))

🔹 Small dataset → using coalesce

  Delta History — retailflow360.bronze.books_data (last 3 ops):
+-------+-------------------+---------+-------------------------------------------------------------------------+
|version|timestamp          |operation|operationParameters                                                      |
+-------+-------------------+---------+-------------------------------------------------------------------------+
|1      |2026-04-22 09:32:57|WRITE    |{mode -> Append, statsOnLoad -> false, partitionBy -> ["ingestion_date"]}|
|0      |2026-04-22 08:40:26|WRITE    |{mode -> Append, statsOnLoad -> false, partitionBy -> ["ingestion_date"]}|
+-------+-------------------+---------+-------------------------------------------------------------------------+


  Delta Detail — retailflow360.bronze.books_data:
+-------------------------------+----------------------------------------------+--------+-----------+----------------+
|name                           |location      

title,description,authors,image,previewlink,publisher,publisheddate,infolink,categories,ratingscount,_raw_file_path,ingestion_timestamp,ingestion_date,batch_id,source_file_name,domain
Its Only Art If Its Well Hung!,null,['Julie Strain'],http://books.google.com/books/content?id=DykPAAAACAAJ&printsec=frontcover&img=1&zoom=1&source=gbs_api,http://books.google.nl/books?id=DykPAAAACAAJ&dq=Its+Only+Art+If+Its+Well+Hung!&hl=&cd=1&source=gbs_api,null,1996,http://books.google.nl/books?id=DykPAAAACAAJ&dq=Its+Only+Art+If+Its+Well+Hung!&hl=&source=gbs_api,['Comics & Graphic Novels'],null,s3://retailflow360-capstone3/raw/books/books_data.csv,2026-04-22T08:40:20.628Z,2026-04-22,ccc4ff1d-cec7-4dab-a79f-fbbaf430d6b7,s3://retailflow360-capstone3/raw/books/books_data.csv,books_data
Dr. Seuss: American Icon,"Philip Nel takes a fascinating look into the key aspects of Seuss's career - his poetry, politics, art, marketing, and place in the popular imagination."" ""Nel argues convincingly that Dr. Seuss is one of the most influential poets in America. His nonsense verse, like that of Lewis Carroll and Edward Lear, has changed language itself, giving us new words like ""nerd."" And Seuss's famously loopy artistic style - what Nel terms an ""energetic cartoon surrealism"" - has been equally important, inspiring artists like filmmaker Tim Burton and illustrator Lane Smith. --from back cover",['Philip Nel'],http://books.google.com/books/content?id=IjvHQsCn_pgC&printsec=frontcover&img=1&zoom=1&edge=curl&source=gbs_api,http://books.google.nl/books?id=IjvHQsCn_pgC&printsec=frontcover&dq=Dr.+Seuss:+American+Icon&hl=&cd=1&source=gbs_api,A&C Black,2005-01-01,http://books.google.nl/books?id=IjvHQsCn_pgC&dq=Dr.+Seuss:+American+Icon&hl=&source=gbs_api,['Biography & Autobiography'],null,s3://retailflow360-capstone3/raw/books/books_data.csv,2026-04-22T08:40:20.628Z,2026-04-22,ccc4ff1d-cec7-4dab-a79f-fbbaf430d6b7,s3://retailflow360-capstone3/raw/books/books_data.csv,books_data
Wonderful Worship in Smaller Churches,"This resource includes twelve principles in understanding small church worship, fifteen practices for planning worship with fewer than 100 people, and suggestions for congregational study.",['David R. Ray'],http://books.google.com/books/content?id=2tsDAAAACAAJ&printsec=frontcover&img=1&zoom=1&source=gbs_api,http://books.google.nl/books?id=2tsDAAAACAAJ&dq=Wonderful+Worship+in+Smaller+Churches&hl=&cd=1&source=gbs_api,null,2000,http://books.google.nl/books?id=2tsDAAAACAAJ&dq=Wonderful+Worship+in+Smaller+Churches&hl=&source=gbs_api,['Religion'],null,s3://retailflow360-capstone3/raw/books/books_data.csv,2026-04-22T08:40:20.628Z,2026-04-22,ccc4ff1d-cec7-4dab-a79f-fbbaf430d6b7,s3://retailflow360-capstone3/raw/books/books_data.csv,books_data
Whispers of the Wicked Saints,"Julia Thomas finds her life spinning out of control after the death of her husband, Richard. Julia turns to her minister for comfort when she finds herself falling for him with a passion that is forbidden by the church. Heath Sparks is a man of God who is busy taking care of his quadriplegic wife who was seriously injured in a sever car accident. In an innocent effort to reach out to a lonely member of his church, Heath finds himself as the man and not the minister as Heath and Julia surrender their bodies to each other and face the wrath of God. Julia finds herself in over her head as she faces a deadly disease, the loss of her home and whispers about her wicked affair. Julia leaves the states offering her body as a living sacrifice in hopes of finding a cure while her heart remains thousands of miles away hoping to one day reunite with the man who holds it hostage.Whispers of the Wicked Saints is a once in a lifetime romance that is breath taking, defying all the rules of romance and bending the laws of love.",['Veronica Haddon'],http://books.google.com/books/content?id=aRSIgJlq6JwC&printsec=frontcover&img=1&zoom=1&source=gbs_api,http://books.google.nl/books?id=aRSIgJlq6JwC&dq=Whispers+of+th

## DOMAIN 4 — BOOKS RATING (user reviews)

#### 1. read

In [0]:
df_books_rating_raw = (
    spark.read
    .format("csv")
    .option("header",      "true")
    .option("inferSchema", "false")
    .option("encoding",    "UTF-8")
    .option("multiLine",   "true")     # review/text spans multiple lines
    .option("quote",       '"')
    .option("escape",      '"')
    .load(S3_RAW_BOOKS_RATING)
)

print(f"  Raw rows loaded : {df_books_rating_raw.count():,}")
print(f"  Raw columns     : {len(df_books_rating_raw.columns)}")
df_books_rating_raw.printSchema()

  Raw rows loaded : 3,000,000
  Raw columns     : 10
root
 |-- Id: string (nullable = true)
 |-- Title: string (nullable = true)
 |-- Price: string (nullable = true)
 |-- User_id: string (nullable = true)
 |-- profileName: string (nullable = true)
 |-- review/helpfulness: string (nullable = true)
 |-- review/score: string (nullable = true)
 |-- review/time: string (nullable = true)
 |-- review/summary: string (nullable = true)
 |-- review/text: string (nullable = true)



#### 2. metadata

In [0]:
df_books_rating_raw = df_books_rating_raw.withColumn("_raw_file_path", col("_metadata.file_path"))

df_books_rating = add_bronze_metadata(
    df_books_rating_raw,
    domain      = "books_rating",
    source_path = S3_RAW_BOOKS_RATING
)
df_books_rating = sanitise_columns(df_books_rating)

print(f"\n  Schema after metadata enrichment:")
df_books_rating.printSchema()



  Schema after metadata enrichment:
root
 |-- id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- price: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- profilename: string (nullable = true)
 |-- review_helpfulness: string (nullable = true)
 |-- review_score: string (nullable = true)
 |-- review_time: string (nullable = true)
 |-- review_summary: string (nullable = true)
 |-- review_text: string (nullable = true)
 |-- _raw_file_path: string (nullable = false)
 |-- ingestion_timestamp: timestamp (nullable = false)
 |-- ingestion_date: date (nullable = false)
 |-- batch_id: string (nullable = false)
 |-- source_file_name: string (nullable = false)
 |-- domain: string (nullable = false)



#### 3. write+ dynamic repartition

In [0]:


# dynamic repartition
size_bytes = get_s3_path_size(S3_RAW_BOOKS_RATING)
partitions = calculate_partitions(size_bytes)
df_books_rating = apply_dynamic_partitioning(df_books_rating, partitions)

# Write external Delta table to S3
books_rating_rows = write_bronze_external(
    df            = df_books_rating,
    table_name    = TBL_BRZ_BOOKS_RATING,
    s3_path       = S3_BRONZE_BOOKS_RATING,
    partition_col = "ingestion_date"
)

# Delta time travel audit
show_delta_history(TBL_BRZ_BOOKS_RATING)
show_table_details(TBL_BRZ_BOOKS_RATING)

print(f"  Sample — Books Rating Bronze:")
display(spark.table(TBL_BRZ_BOOKS_RATING).limit(5))


  Delta History — retailflow360.bronze.books_rating (last 3 ops):
+-------+-------------------+---------+-------------------------------------------------------------------------+
|version|timestamp          |operation|operationParameters                                                      |
+-------+-------------------+---------+-------------------------------------------------------------------------+
|1      |2026-04-22 09:35:08|WRITE    |{mode -> Append, statsOnLoad -> false, partitionBy -> ["ingestion_date"]}|
|0      |2026-04-22 08:42:26|WRITE    |{mode -> Append, statsOnLoad -> false, partitionBy -> ["ingestion_date"]}|
+-------+-------------------+---------+-------------------------------------------------------------------------+


  Delta Detail — retailflow360.bronze.books_rating:
+---------------------------------+------------------------------------------------+--------+-----------+----------------+
|name                             |location                             

id,title,price,user_id,profilename,review_helpfulness,review_score,review_time,review_summary,review_text,_raw_file_path,ingestion_timestamp,ingestion_date,batch_id,source_file_name,domain
B000092PU5,"The Power of Full Engagement: Managing Energy, Not Time, Is the Key to Performance and Personal Renewal",null,A1759D33T0DZH,"Marco Antonio G. Darvas ""Marco A Darvas""",0/0,5.0,1200614400,New approach for an old theme,"I believe the author's are very fortunate to write about a subject so necessary in our days.I loved the approach.I work in Brazil as a personal Coach for Productive Principles, and will use this learning on my Lectures.",s3://retailflow360-capstone3/raw/books/Books_rating.csv,2026-04-22T08:41:21.422Z,2026-04-22,ccc4ff1d-cec7-4dab-a79f-fbbaf430d6b7,s3://retailflow360-capstone3/raw/books/Books_rating.csv,books_rating
B000092PU5,"The Power of Full Engagement: Managing Energy, Not Time, Is the Key to Performance and Personal Renewal",null,A1UDO2272HS7RU,"F. Cubillos ""MD Reader""",4/12,3.0,1171756800,Not as expected,"I bought this book as a requirement of a Leadeship course. This is a self-help book with some interesting advice, but repetitive, not very well written, and many common sense advice as well. Overall, don't expect much out of this book.",s3://retailflow360-capstone3/raw/books/Books_rating.csv,2026-04-22T08:41:21.422Z,2026-04-22,ccc4ff1d-cec7-4dab-a79f-fbbaf430d6b7,s3://retailflow360-capstone3/raw/books/Books_rating.csv,books_rating
B000092PU5,"The Power of Full Engagement: Managing Energy, Not Time, Is the Key to Performance and Personal Renewal",null,A2E2CS4XEBKKR7,Derrik Heldstein,2/9,5.0,1048636800,Maximum Managment Baby!,"I'll keep this brief, as I am now practicing what I have learned. Jim and Tony have really told it like it needed to be told! Indeed, how can one expect to achieve personal renewal without energy? I have truly been transformed by this reading experience, and can now shoot bolts of electricity from my fingertips. That's energy baby!",s3://retailflow360-capstone3/raw/books/Books_rating.csv,2026-04-22T08:41:21.422Z,2026-04-22,ccc4ff1d-cec7-4dab-a79f-fbbaf430d6b7,s3://retailflow360-capstone3/raw/books/Books_rating.csv,books_rating
B000092PU5,"The Power of Full Engagement: Managing Energy, Not Time, Is the Key to Performance and Personal Renewal",null,A38QDASP9OKG4S,"BJ Wright ""seointern""",0/0,5.0,1223078400,Focus On Our Energy Not Time Management,"This book is awesome, and the concepts are so true and helpful. There are way to many books out there to help manage our time, when if we focused on our energy we would accomplish so much more in day. The book covers a number of different aspects that help you become effective in your daily tasks, each one of them could fill a full book alone.",s3://retailflow360-capstone3/raw/books/Books_rating.csv,2026-04-22T08:41:21.422Z,2026-04-22,ccc4ff1d-cec7-4dab-a79f-fbbaf430d6b7,s3://retailflow360-capstone3/raw/books/Books_rating.csv,books_rating
B000092PU5,"The Power of Full Engagement: Managing Energy, Not Time, Is the Key to Performance and Personal Renewal",null,A3TNTR0KSR2P4C,Hide Harashima,14/22,2.0,1081728000,lots of fluff,"This is a book about managing and developing your energy and apply it to life and work. Every person has different biorhythms, and the concept is that to accomplish what you want, you need to train your body to maximize your energy when you need it, just as an athlete trains everyday to get the job done when it counts. The Power of Full Engagement was full of common sense and anecdotal stories of how other people 'trained' to manage their energy (fluff).There was nothing new in the book, if you have read other self-help or motivation books. I also felt like this book was written to promote the author's institute that trained several high-profile athletes too many times.",s3://retailflow360-capstone3/raw/books/Books_rating.csv,2026-04-22T08:41:21.422Z,2026-04-22,ccc4ff1d-cec7-4dab-a79f-fbbaf430d6b7,s3://retailflow360-capstone3

## Verify

In [0]:
print(f"\n  Catalog verification:")
for tbl in [TBL_BRZ_ELECTRONICS, TBL_BRZ_LIQUOR, TBL_BRZ_BOOKS_DATA, TBL_BRZ_BOOKS_RATING]:
    cnt = spark.table(tbl).count()
    print(f"    ✅ {tbl} → {cnt:,} rows")


  Catalog verification:
    ✅ retailflow360.bronze.electronics → 373,700 rows
    ✅ retailflow360.bronze.liquor_sales → 39,333,526 rows
    ✅ retailflow360.bronze.books_data → 424,808 rows
    ✅ retailflow360.bronze.books_rating → 6,000,000 rows
